# 06 Baseline Model Training

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/unet-ensemble')

## Dataset Preparation

In [ ]:
!pip install safetensors huggingface_hub segmentation-models-pytorch

In [ ]:
import os
import copy
import torch
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

from safetensors.torch import save_file
from torch.utils.data import DataLoader
from src.dataset.rgb_dataset import RGBDataset, RGBDataLoader
from src.training.baseline import RGBBaseline, Train
from src.utils.checkpoint_manager import CheckpointManager

In [ ]:
from huggingface_hub import login
token=''
login(token=token)

In [ ]:
IMG_SIZE = 256
DATASET_ROOT = '/content/drive/Shareddrives/U-Net Ensemble/Dataset'
BATCH_SIZE  = 8
LR          = 1e-4
EPOCHS      = 50
PATIENCE    = 10           # early stopping patience
CHECKPOINT  = 'best_diff_id.pth'
CHECKPOINT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/checkpoint/Diff-ID'
RESUME_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'resume_checkpoint.pth')
NUM_WORKERS = 2

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
loader = RGBDataLoader(mask_folder  = "GTA - Masks",
                rgb_folder          = "Normalized",
                categories          = ['Synthetic', 'Authentic'],
                templates           = ['template-a', 'template-albania', 'template-b',
                                        'template-c', 'template-chile', 'template-deutschland',
                                        'template-spain', 'template-usa'])

In [ ]:
train_samples = loader.load_images('Training', DATASET_ROOT)
val_samples   = loader.load_images('Validation', DATASET_ROOT)

train_ds = RGBDataset(train_samples, IMG_SIZE, augment=True)
val_ds   = RGBDataset(val_samples,   IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Model

### XceptionNet

In [ ]:
xceptionnet = smp.Unet(
    encoder_name   ='xception',
    encoder_weights='imagenet',
    in_channels    =3,
    classes        =1,
    activation     =None,
)

In [ ]:
model = RGBBaseline(model=xceptionnet).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

### Diff-ID

In [ ]:
diffid = smp.Unet(
    encoder_name    ='resnet34',
    encoder_weights ='imagenet',
    in_channels     =3,
    classes         =1,
    activation      =None,
)

In [ ]:
model = MBENBaseline(model=diffid).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

### Two-Stream CNN

In [ ]:
twostream = smp.Unet(
    encoder_name    ='efficientnet-b0',
    encoder_weights ='imagenet',
    in_channels     =3,
    classes         =1,
    activation      =None
)

In [ ]:
MBEN_OUT_CHANNELS = 64

model = RGBBaseline(model=twostream).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## Loss, Optimizer, Scheduler

In [ ]:
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

## Checkpoint Utilities

In [ ]:
train = Train(device, dice_loss)

checkpoint = CheckpointManager()

# ── Load checkpoint if one exists (resumes automatically) ──
(
    start_epoch,
    best_val_loss,
    early_stop_counter,
    train_losses,
    val_losses,
    val_auc,
    val_precision,
    val_recall,
    val_mcc
) = checkpoint.load_checkpoint(model, optimizer, scheduler, RESUME_CHECKPOINT)

best_model_wts = copy.deepcopy(model.state_dict())

## Training Loop

In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss  = train.run_epoch(train_loader, model, optimizer, train=True)
    val_metrics = train.run_epoch(val_loader,   model, train=False)
    val_loss = val_metrics["val_loss"]

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_auc.append(val_metrics["AUC_ROC"])
    val_precision.append(val_metrics["Precision"])
    val_recall.append(val_metrics["Recall"])
    val_mcc.append(val_metrics["MCC"])

    print(
        f'Epoch [{epoch:03d}/{EPOCHS}]  '
        f'Train Loss: {train_loss:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'AUC: {val_metrics["AUC_ROC"]:.4f} | '
        f'Precision: {val_metrics["Precision"]:.4f} | '
        f'Recall: {val_metrics["Recall"]:.4f} | '
        f'MCC: {val_metrics["MCC"]:.4f}'
    )

    if val_loss < best_val_loss:
        best_val_loss      = val_loss
        best_model_wts     = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
        save_file({k: v.cpu() for k, v in best_model_wts.items()}, f'{CHECKPOINT_DIR}/model.safetensors')
        print(f'  ✔ New best val loss: {best_val_loss:.4f} — model.safetensors saved.')
    else:
        early_stop_counter += 1
        print(f'  No improvement ({early_stop_counter}/{PATIENCE})')
        if early_stop_counter >= PATIENCE:
            print('Early stopping triggered.')
            break

    # ── Save resume checkpoint to Drive every epoch ──
    checkpoint.save_checkpoint(
        model, optimizer, scheduler, epoch,
        best_val_loss, early_stop_counter,
        train_losses, val_losses, val_auc,
        val_precision, val_recall, val_mcc,
        RESUME_CHECKPOINT,
    )

model.load_state_dict(best_model_wts)
print(f'\nTraining complete. Best Val Loss: {best_val_loss:.4f}')

## Loss Curves

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train Dice Loss')
plt.plot(val_losses,   label='Val Dice Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MBEN + XceptionNet Training & Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()

## Quick Inference Check

In [ ]:
model.eval()
rgb, masks = next(iter(val_loader))
rgb = rgb.to(device)
with torch.no_grad():
    preds = torch.sigmoid(model(rgb)).cpu().numpy()

## Save Final Model

### Save to Hugging Face Repository

In [ ]:
from huggingface_hub import HfApi, login

login(token="write-token")

api = HfApi()

# Create the repo first (only needed once)
username = 'hf-username'
api.create_repo(repo_id=f"{username}/mben-baselines", exist_ok=True)

# Upload the weights
api.upload_file(
    path_or_fileobj=f"{CHECKPOINT_DIR}/model.safetensors",
    path_in_repo='xceptionnet/model.safetensors',
    repo_id=f"{username}/mben-baselines",
)

In [ ]:
import json

config = {
    "model_type": "XceptionNet",
    "encoder": "xceptionnet", # change according to model
    "encoder_weights": "imagenet",
    "in_channels": 3,
    "classes": 1,
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

api.upload_file(
    path_or_fileobj="config.json",
    path_in_repo="xceptionet/config.json",
    repo_id=f"{username}/mben-baselines",
)